In [1]:
library(circlize)
library(dplyr)

circlize version 0.4.15
CRAN page: https://cran.r-project.org/package=circlize
Github page: https://github.com/jokergoo/circlize
Documentation: https://jokergoo.github.io/circlize_book/book/

If you use it in published research, please cite:
Gu, Z. circlize implements and enhances circular visualization
  in R. Bioinformatics 2014.

This message can be suppressed by:
  suppressPackageStartupMessages(library(circlize))



载入程辑包：‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
load("../snp_matrix_2860770.RData")

In [6]:
dim(snp_matrix)

[1]     400 2860770

In [13]:
col_names <- colnames(snp_matrix)

In [15]:
#=============== 第三部分：提取SNP位置信息 ===============

# 从SNP名称中提取染色体编号
# 假设SNP名称格式为"chr_pos_allele"，提取第一部分作为染色体编号
col1 <- as.integer(gsub("^(\\d+)_.*$", "\\1", col_names))

# 从SNP名称中提取物理位置
# 提取第二部分作为染色体上的物理位置
col2 <- as.integer(gsub("^\\d+_(\\d+)_.*$", "\\1", col_names))

# 创建SNP信息数据框，包含染色体和位置信息
pos_info <- data.frame(chr = col1, pos = col2)

In [17]:
dim(pos_info)
head(pos_info)

[1] 2860770       2

,chr,pos
,<int>,<int>
1,1,1749
2,1,1804
3,1,1872
4,1,1994
5,1,2173
6,1,2215


In [19]:
save(pos_info, file="./data/pos_info.RData")

In [20]:
# ==============================================================================
# 1. 数据预处理 (基于你已有的 pos_info)
# ==============================================================================
# 确保 pos_info 存在
if (!exists("pos_info")) {
  stop("错误: 找不到名为 'pos_info' 的数据框，请先加载你的数据。")
}

# 1.1 规范化列名和格式
# 假设第一列是染色体(数字)，第二列是位置
# 为了安全起见，我们显式重命名列，不破坏原始数据
plot_data <- pos_info
colnames(plot_data) <- c("Chr", "Pos")

# 1.2 优化染色体排序 (关键步骤)
# 如果直接用字符，顺序会变成 1, 10, 2... 
# 我们先提取所有唯一的染色体编号，按数字大小排序
chr_order_num <- sort(unique(as.numeric(as.character(plot_data$Chr))))
# 加上 "chr" 前缀 (例如 chr1, chr2)，符合学术图表习惯
chr_labels <- paste0("chr", chr_order_num)

# 将 Chr 列转换为因子(Factor)，并指定正确的 Level 顺序
plot_data$Chr <- factor(paste0("chr", plot_data$Chr), levels = chr_labels)

# ==============================================================================
# 2. 计算绘图参数
# ==============================================================================
print("正在计算染色体长度...")
# 2.1 定义染色体长度 (Karyotype)
# 根据每个染色体上 SNP 的最大位置来确定染色体长度
chrom_info <- plot_data %>%
  group_by(Chr) %>%
  summarise(Start = 0, End = max(Pos)) %>%
  as.data.frame()

print("正在计算 SNP 密度 (这可能需要几秒钟)...")
# 2.2 计算 SNP 密度 (Window Size = 1Mb)
window_size <- 1000000 # 1Mb 窗口
snp_density <- plot_data %>%
  mutate(Window_Start = floor(Pos / window_size) * window_size) %>%
  group_by(Chr, Window_Start) %>%
  summarise(Count = n(), .groups = 'drop') %>%
  mutate(Window_End = Window_Start + window_size) %>%
  select(Chr, Window_Start, Window_End, Count) %>%
  as.data.frame()

# 检查一下密度数据的最大值，防止颜色过饱和
max_density <- max(snp_density$Count)
print(paste("1Mb窗口内的最大SNP数量为:", max_density))

# ==============================================================================
# 3. 开始绘图
# ==============================================================================
# 输出为 PDF
pdf("Figure1_Genome_Circos.pdf", width = 8, height = 8)

# 初始化 Circos 参数
circos.clear()
# start.degree=90 让 chr1 从 12 点钟方向开始
# gap.degree 设置染色体之间的间隙，最后一个参数 5 是为了留出位置给图例或结束
circos.par("start.degree" = 90, gap.degree = c(rep(1, nrow(chrom_info))))

# 3.1 初始化基因组外圈 (染色体骨架)
circos.genomicInitialize(chrom_info, 
                         plotType = c("axis", "labels"), # 显示刻度和标签
                         major.by = 50000000) # 每 50Mb 一个大刻度

# 3.2 添加 SNP 密度轨道
# 动态调整颜色映射：根据你的数据实际密度来设置
# 这里设置：0(白色) -> 中位数(天蓝) -> 95%分位数(红色)，这样对比度最好
density_quantile <- quantile(snp_density$Count, probs = 0.95)
col_fun = colorRamp2(c(0, density_quantile/2, density_quantile), c("white", "skyblue", "red"))

circos.genomicTrack(snp_density, ylim = c(0, 1), panel.fun = function(region, value, ...) {
  circos.genomicRect(region, value, col = col_fun(value[[1]]), border = NA)
}, track.height = 0.15, bg.border = NA)

# 3.3 添加图例说明 (放在圆圈中心)
text(0, 0, paste0("SNP Density\n(Max: ", max_density, "/Mb)"), cex = 1.2, font = 2)

dev.off()
print("绘图完成！文件已保存为 Figure1_Genome_Circos.pdf")

[1] "正在计算染色体长度..."
[1] "正在计算 SNP 密度 (这可能需要几秒钟)..."
[1] "1Mb窗口内的最大SNP数量为: 2282"


Note: 2 points are out of plotting region in sector 'chr1', track '2'.

Note: 2 points are out of plotting region in sector 'chr2', track '2'.

Note: 2 points are out of plotting region in sector 'chr3', track '2'.

Note: 2 points are out of plotting region in sector 'chr4', track '2'.

Note: 2 points are out of plotting region in sector 'chr5', track '2'.

Note: 2 points are out of plotting region in sector 'chr6', track '2'.

Note: 2 points are out of plotting region in sector 'chr7', track '2'.

Note: 2 points are out of plotting region in sector 'chr8', track '2'.

Note: 2 points are out of plotting region in sector 'chr9', track '2'.

Note: 2 points are out of plotting region in sector 'chr10', track '2'.



png 
  2

[1] "绘图完成！文件已保存为 Figure1_Genome_Circos.pdf"
